# Retrieval over technical notes

**Technical report 3 of the switchgear-estimation-assistant project**

## Purpose

The agent answers questions about parts with tools. Some questions are not about prices. They ask what a part does. A set of short technical notes answers these. This report describes how the notes are searched, how the search was tested, and where it fails.

All notes are synthetic.    


## Design

`ai_engine/estimation_assistant/retrieval.py` holds four short notes (D1 to D4) and one tool, `search_notes`.

1. Each note is turned into a list of numbers (an embedding) by an embedding model.
2. The question is turned into a list of numbers by the same model.
3. Each note is scored against the question with cosine similarity: the dot product of the two lists, divided by the product of their lengths. A higher score means a closer meaning.
4. The note with the highest score is returned as `Source <id> (score <value>): <text>`.

One embedding covers a whole text, not each word. The list length is fixed by the model: 768 numbers for `nomic-embed-text`, 1024 for `mxbai-embed-large`.

Notes and questions must be embedded by the same model. Lists from different models cannot be compared.

## Test setup

Two embedding models were compared on the same four notes and two questions.

- Easy question: shares words with its correct note.
- Hard question: a paraphrase that shares no key words with its correct note (D3).

A model passes when the correct note has the highest score. The gap between the first and second score shows how safe the result is.

The nomic model was also tested with its task labels (`search_document:` and `search_query:`).

## Results

| Model | Question | Top note | Top score | Correct |
|---|---|---|---|---|
| nomic-embed-text, with labels | Easy | D2 | 0.833 | Yes (gap 0.163) |
| nomic-embed-text | Hard | D1 | 0.578 | No (D3 scored 0.574) |
| nomic-embed-text, with labels | Hard | D2 | 0.583 | No (D1 0.582, D3 0.581) |
| mxbai-embed-large | Easy | D2 | 0.858 | Yes (gap 0.155) |
| mxbai-embed-large | Hard | D3 | 0.699 | Yes (gap 0.122) |

The saved output of the nomic Hard run without labels (D1 0.578, D3 0.574) did not survive in the notebook, so that row cannot be checked there.

The task labels did not fix the hard question for the nomic model. `mxbai-embed-large` was chosen because it answered both questions correctly with a clear gap.

Four notes and two questions are a small test. The result shows that the models differ on paraphrases. It does not give an accuracy figure.

## Limits

- The search always returns the nearest note, even when no note fits. A low score is the only sign. A refusal step that reports no source exists only in a notebook prototype. In the project, `sources` lists the note even if the model declines to answer from it.
- All four notes are embedded on every call. Caching them would save time and is a planned change.
- Only one note is returned. A question that needs two notes is answered from one.
- The agent rewrites the question before calling the tool, so the text that is embedded can differ from the user's wording.
- The tool description tells the model not to use `search_notes` for prices. The model can still choose the wrong tool.

## Connection to the rest of the project

`search_notes` is registered in `TOOLS` in `agent.py`. The run result lists the note ids in `sources`, filled by code from the run log. The number check in `checks.py` treats the note text as evidence for numbers in the answer.